Imports and paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor

In [2]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
raw_data_dir = ROOT / "data" / "raw"
processed_data_dir = ROOT / "data" / "processed"
reports_figures = ROOT / "reports" / "figures"

Loading the data

In [3]:
pl_players_2025_26 = pd.read_csv(processed_data_dir / "pl_players_2025_26.csv")
pl_players_2025_26.tail()

,player_id,name,club_id,club,position,sub_position,foot,height_in_cm,international_caps,age,contract_years_left,appearances,minutes_played,goals,assists,yellow_cards,value_start_eur,value_end_eur,value_change_eur,value_change_pct
563,1141628,Yunus Konak,1148,Brentford FC,Midfield,Defensive Midfield,right,181.0,16.0,20.4,3.1,0,0,0,0,0,4500000.0,4000000.0,-500000.0,-11.1
564,1193867,Pedro Lima,543,Wolverhampton Wanderers,Defender,Right-Back,right,174.0,1.0,19.9,3.1,9,259,0,0,0,5000000.0,4000000.0,-1000000.0,-20.0
565,1230661,Mateus Mané,543,Wolverhampton Wanderers,Midfield,Attacking Midfield,right,175.0,2.0,18.7,3.1,27,1790,3,2,0,200000.0,20000000.0,19800000.0,9900.0
566,1283997,Diego León,985,Manchester United,Defender,Left-Back,left,177.0,1.0,19.1,3.1,0,0,0,0,0,4000000.0,4000000.0,0.0,0.0
567,1322090,Jocelin Ta Bi,289,Sunderland AFC,Attack,Right Winger,left,175.0,3.0,21.3,4.1,1,39,0,0,0,125000.0,3000000.0,2875000.0,2300.0


## Data preparation

The features chosen to explain the market value change.

In [4]:
features = ["position", "age", "contract_years_left", "minutes_played", "goals", "assists", "value_start_eur"]
model_data = pl_players_2025_26[features].copy()

37 players have no contract end date : they are mostly old players who barely play and whose contract is ending, so their missing contract is replaced by 0 (contract over) instead of removing them.

In [5]:
pl_players_2025_26[pl_players_2025_26["contract_years_left"].isna()].shape

(37, 20)

In [6]:
model_data["contract_years_left"] = model_data["contract_years_left"].fillna(0)

The target is the log-ratio of the market value: log(value at the end / value at the start). A doubled value gives +0.69 and a value divided by two gives -0.69.

In [7]:
model_data["value_change_log"] = np.log(pl_players_2025_26["value_end_eur"] / pl_players_2025_26["value_start_eur"])

The value at the start of the season goes from 0.1 M€ to 180 M€. It is put in log so that the model compares the values in proportion: going from 1 to 2 M€ counts as much as going from 50 to 100 M€.

In [8]:
model_data["value_start_log"] = np.log(model_data["value_start_eur"])
model_data = model_data.drop(columns="value_start_eur")

The position becomes 4 columns with 0 or 1, one per position.

In [9]:
model_data = pd.get_dummies(model_data, columns=["position"], dtype=int)

Check that there are no missing values left.

In [10]:
model_data.isna().sum()

age                    0
contract_years_left    0
minutes_played         0
goals                  0
assists                0
value_change_log       0
value_start_log        0
position_Attack        0
position_Defender      0
position_Goalkeeper    0
position_Midfield      0
dtype: int64

In [11]:
model_data.shape

(568, 11)

## Features and target

In [12]:
y = model_data["value_change_log"]
X = model_data.drop(columns="value_change_log")

## Train / test split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=10)

## Linear regression model

In [14]:
model = LinearRegression().fit(X_train, y_train)

model_coefficients = pd.DataFrame(
    model.coef_, 
    index=model.feature_names_in_, 
    columns=["coefficient"]
)
model_coefficients.index.name = "features"

## Model evaluation

### On the test set

In [15]:
y_predicted = model.predict(X_test)

In [16]:
mae = mean_absolute_error(y_test, y_predicted)
print(f"Mean Absolute Error (MAE) : {mae:.4f}")
print(f"Approximate percentage error: {(np.exp(mae) - 1) * 100:.2f}%")

Mean Absolute Error (MAE) : 0.2710
Approximate percentage error: 31.12%


In [17]:
r2 = r2_score(y_test, y_predicted)
print(f"R² score on the test dataset : {r2:.4f}")

R² score on the test dataset : 0.5605


### On the training set

We can test the model behavior by looking at its predictions on the training set it was trained on.

In [18]:
y_predicted_train = model.predict(X_train)
r2 = r2_score(y_train, y_predicted_train)
print(f"R² score on the training dataset : {r2:.4f}")

R² score on the training dataset : 0.5277


The score is slightly better on the test set (0.56) than on the training set (0.53), even though we would have expected the opposite.

The two scores are close, so the model does not overfit.

### Baseline

In [19]:
baseline = DummyRegressor().fit(X_train, y_train)

y_predicted_baseline = baseline.predict(X_test)

mae = mean_absolute_error(y_test, y_predicted_baseline)
print("------- Baseline Model -------")
print(f"Mean Absolute Error (MAE)  : {mae:.4f}")
print(f"Approximate percentage error: {(np.exp(mae) - 1) * 100:.2f}%")
r2 = r2_score(y_test, y_predicted_baseline)
print(f"R² score : {r2:.4f}")

------- Baseline Model -------
Mean Absolute Error (MAE)  : 0.3845
Approximate percentage error: 46.89%
R² score : -0.0056


The baseline makes bigger errors than the model (about 47% against 31%), and its R² is close to 0 : predicting the mean explains nothing.

The model makes significantly fewer mistakes than the dummy model. The model is clearly learning something.

## Interpretation of the coefficients

In [20]:
model_coefficients

,coefficient
features,
age,-0.066345
contract_years_left,0.075150
minutes_played,0.000262
goals,0.007932
assists,-0.000480
value_start_log,-0.266608
position_Attack,0.091307
position_Defender,0.011849
position_Goalkeeper,-0.118245


The coefficients are in log-ratio. They are converted to % with exp(coefficient) - 1.

This % can be read directly for the age, the contract, the goals and the assists (effect of +1). The minutes played, the starting value and the positions need another unit (see below).

In [21]:
def val_change(coeff):
    return (np.exp(coeff) - 1) * 100

model_coefficients["value_change_pct"] = val_change(model_coefficients["coefficient"])
model_coefficients

,coefficient,value_change_pct
features,,
age,-0.066345,-6.419237
contract_years_left,0.075150,7.804634
minutes_played,0.000262,0.026214
goals,0.007932,0.796367
assists,-0.000480,-0.047963
value_start_log,-0.266608,-23.402687
position_Attack,0.091307,9.560537
position_Defender,0.011849,1.191923
position_Goalkeeper,-0.118245,-11.152122


### Minutes played

The coefficient of the minutes played (0.000262) is the effect of one more minute: it is too small to be read, so a bigger unit is used.

In [22]:
Q1 = model_data["minutes_played"].quantile(0.25)
Q3 = model_data["minutes_played"].quantile(0.75)

IQR = Q3 - Q1

print(f"Q1 : {Q1:.2f} minutes played")
print(f"Q3 : {Q3:.2f} minutes played")
print(f"Interquartile range : {IQR:.2f} minutes played")

Q1 : 270.00 minutes played
Q3 : 2140.75 minutes played
Interquartile range : 1870.75 minutes played


A descriptive analysis of minutes played reveals significant variation among the players in the sample, with a first quartile of 270 minutes and a third quartile of 2,140.75 minutes, for an interquartile range of 1,870.75 minutes. 

To simplify the economic interpretation of the estimated coefficients, we use a unit of 1,800 minutes, which corresponds to 20 full games and approximately represents the observed interquartile range. This choice allows us to express the results on a standardised scale which is consistent with the variation in minutes played within the sample.

In [23]:
effect_1800_minutes = model_coefficients.loc["minutes_played", "coefficient"] * 1800
print(f"+1,800 minutes played : {val_change(effect_1800_minutes):.1f} % value change")

+1,800 minutes played : 60.3 % value change


### Assists

How can we explain the coefficient given to the assists ? Why is it negative ?

Let's look at a model without the minutes played and a model without the goals.

In [24]:
m = model_coefficients.loc["assists", "coefficient"]

model_no_minutes = LinearRegression().fit(X_train.drop(columns="minutes_played"), y_train)
model_no_minutes_assists = pd.Series(model_no_minutes.coef_, index=model_no_minutes.feature_names_in_)["assists"]

model_no_goals = LinearRegression().fit(X_train.drop(columns="goals"), y_train)
model_no_goals_assists = pd.Series(model_no_goals.coef_, index=model_no_goals.feature_names_in_)["assists"]

print(f"Assists coefficient for the 'full' model: {m}")
print(f"Assists coefficient without minutes played: {model_no_minutes_assists}")
print(f"Assists coefficient without goals: {model_no_goals_assists}")

Assists coefficient for the 'full' model: -0.0004797472511116507
Assists coefficient without minutes played: 0.035217005630765845
Assists coefficient without goals: 0.0016844078674491858


Now we get a positive coefficient for the assists, which is more 'logical'. We can conclude that the negative coefficient of the assists in the 'full' model is due to the correlation between the minutes played, the goals and the assists. Just as we saw in the correlation matrix in the previous notebook.

With goals and minutes played, the assists give no additional information to the model, that's why we got a negative coefficient.

### Value at the start of the season (log)

The coefficient of the starting value is negative : given the same season, the more expensive a player is at the start, the less his value grows.

### Positions

In [25]:
positions = pl_players_2025_26["position"].unique()

In [26]:
def compare_positions(base, other):
    """Compared to `base`, how much does `other` progress (in %)?"""
    diff = (
        model_coefficients.loc[f"position_{other}", "coefficient"]
        - model_coefficients.loc[f"position_{base}", "coefficient"]
    )
    change = val_change(diff)
    print(f"Compared to {base}, {other} progresses by {change:.1f} %")
    return change

for i, pos1 in enumerate(positions):
    for pos2 in positions[i + 1:]:
        compare_positions(pos1, pos2)


Compared to Midfield, Goalkeeper progresses by -12.5 %
Compared to Midfield, Defender progresses by -0.3 %
Compared to Midfield, Attack progresses by 7.9 %
Compared to Goalkeeper, Defender progresses by 13.9 %
Compared to Goalkeeper, Attack progresses by 23.3 %
Compared to Defender, Attack progresses by 8.3 %


Given the same age, contract, minutes played, goals, assists and starting value, a forward's value increases by about 8% more than that of a midfielder.

## Conclusion

- **The model works**: it explains 56% of the value change (R² of 0.56, against 0 for the baseline), with an error of about 31%. The rest comes from what the data does not show (injuries, transfers, hype...).
- **Playing time is the strongest factor**: 1,800 more minutes played (about 10 matches) give about +60%. Once the minutes are known, the goals add little (+0.8% per goal) and the assists nothing.
- **Age and contract** go in opposite directions: about −6% per year of age, +8% per year of contract left.
- **Starting value**: the most expensive players grow the least.
- **Position**: forwards grow the most and goalkeepers the least ; defenders and midfielders are the same.

The model explains the season afterwards, it does not predict it: the minutes, goals and assists are only known at the end of the season.